In [34]:
from dotenv import load_dotenv
load_dotenv()
from pprint import pprint

from langchain.agents import create_agent
from langchain.agents import AgentState
from langchain.tools import tool
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage
from langchain.messages import ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


In [20]:
class CustomState(AgentState):
    favourite_colour: str

In [21]:
# Write the state

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [22]:
# Create the agent

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [23]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response["messages"][-1].content)

('Great — I’ve saved green as your favourite colour. Want me to tailor future '
 'suggestions around green (palettes, decor ideas, product recommendations, '
 'etc.)? If you’d like, you can share any other preferences to set as well.')


In [25]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response["messages"][-1].content)

"I'm good, thanks for asking! How can I help you today?"


In [29]:
# Read State

@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

In [30]:
agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState,
)

In [31]:
response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [ ]:
pprint(response["messages"][-1].content)

("Nice—I've saved green as your favourite colour.\n"
 '\n'
 'Would you like me to:\n'
 '- confirm the current stored value (show it back to you),\n'
 '- use this preference to tailor color suggestions or themes (e.g., green '
 'palettes, outfits, backgrounds),\n'
 '- or update it to something else?')


In [33]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response["messages"][-1].content)

('Your favourite colour is green.\n'
 '\n'
 'Would you like me to tailor color suggestions or themes around green, or '
 'would you like to update it to something else?')
